In [ ]:
# SkyVision - Model Training Notebook
# This notebook trains both LSTM and Prophet models

# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from prophet import Prophet
import joblib
import os

print("Libraries imported successfully!")

# Cell 2: Load and Preprocess Data
def load_weather_data(file_path):
    """Load weather data from CSV"""
    df = pd.read_csv(file_path)
    
    # Basic data cleaning
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    
    # Handle missing values
    numeric_columns = ['temperature', 'humidity', 'wind_speed', 'cloud_cover', 'rainfall']
    for col in numeric_columns:
        if col in df.columns:
            df[col].fillna(df[col].median(), inplace=True)
    
    return df

# Load your dataset
# df = load_weather_data('data/indonesia_weather.csv')
# df.head()

print("Data loading function ready!")

# Cell 3: Data Visualization
def plot_weather_data(df):
    """Plot weather data distributions"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Temperature over time
    axes[0, 0].plot(df['timestamp'][:168], df['temperature'][:168])
    axes[0, 0].set_title('Temperature (First Week)')
    axes[0, 0].set_xlabel('Time')
    axes[0, 0].set_ylabel('Temperature (°C)')
    
    # Humidity distribution
    axes[0, 1].hist(df['humidity'], bins=30, alpha=0.7)
    axes[0, 1].set_title('Humidity Distribution')
    axes[0, 1].set_xlabel('Humidity (%)')
    axes[0, 1].set_ylabel('Frequency')
    
    # Wind speed boxplot
    axes[1, 0].boxplot(df['wind_speed'])
    axes[1, 0].set_title('Wind Speed Distribution')
    axes[1, 0].set_ylabel('Wind Speed (km/h)')
    
    # Cloud cover over time
    axes[1, 1].scatter(df['temperature'][:1000], df['cloud_cover'][:1000], alpha=0.5)
    axes[1, 1].set_title('Temperature vs Cloud Cover')
    axes[1, 1].set_xlabel('Temperature (°C)')
    axes[1, 1].set_ylabel('Cloud Cover (%)')
    
    plt.tight_layout()
    plt.show()

# plot_weather_data(df)

print("Visualization function ready!")

# Cell 4: Prepare Data for LSTM
def prepare_lstm_data(df, sequence_length=24):
    """Prepare data for LSTM model"""
    features = ['temperature', 'humidity', 'wind_speed', 'cloud_cover', 'hour', 'day_of_week']
    targets = ['temperature', 'humidity', 'wind_speed', 'cloud_cover']
    
    # Scale features and targets
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X = df[features].values
    y = df[targets].values
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    # Create sequences
    X_seq, y_seq = [], []
    for i in range(len(X_scaled) - sequence_length):
        X_seq.append(X_scaled[i:i+sequence_length])
        y_seq.append(y_scaled[i+sequence_length])
    
    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    
    # Split data
    train_size = int(0.8 * len(X_seq))
    X_train, X_test = X_seq[:train_size], X_seq[train_size:]
    y_train, y_test = y_seq[:train_size], y_seq[train_size:]
    
    return X_train, X_test, y_train, y_test, scaler_X, scaler_y

print("LSTM data preparation function ready!")

# Cell 5: Build and Train LSTM
def build_and_train_lstm(X_train, y_train, X_test, y_test, epochs=50):
    """Build and train LSTM model"""
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(4)  # 4 output features
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=1
    )
    
    return model, history

# Example usage:
# X_train, X_test, y_train, y_test, scaler_X, scaler_y = prepare_lstm_data(df)
# model, history = build_and_train_lstm(X_train, y_train, X_test, y_test)

print("LSTM training function ready!")

# Cell 6: Evaluate LSTM Model
def evaluate_lstm(model, history, X_test, y_test, scaler_y):
    """Evaluate LSTM model performance"""
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(history.history['loss'], label='Training Loss')
    axes[0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].plot(history.history['mae'], label='Training MAE')
    axes[1].plot(history.history['val_mae'], label='Validation MAE')
    axes[1].set_title('Model MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_original = scaler_y.inverse_transform(y_pred)
    y_test_original = scaler_y.inverse_transform(y_test)
    
    # Calculate metrics
    mse = np.mean((y_test_original - y_pred_original) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_test_original - y_pred_original))
    
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    
    return y_pred_original, y_test_original

# y_pred, y_true = evaluate_lstm(model, history, X_test, y_test, scaler_y)

print("LSTM evaluation function ready!")

# Cell 7: Save LSTM Model
def save_lstm_model(model, scaler_X, scaler_y, path='../model/saved'):
    """Save LSTM model and scalers"""
    os.makedirs(path, exist_ok=True)
    
    model.save(f'{path}/lstm_model.h5')
    joblib.dump(scaler_X, f'{path}/scaler_X.pkl')
    joblib.dump(scaler_y, f'{path}/scaler_y.pkl')
    
    print(f"Model saved to {path}")

# save_lstm_model(model, scaler_X, scaler_y)

print("Save function ready!")

# Cell 8: Prepare and Train Prophet Models
def train_prophet_models(df):
    """Train Prophet models for each feature"""
    features = ['temperature', 'humidity', 'wind_speed', 'cloud_cover']
    models = {}
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    for idx, feature in enumerate(features):
        print(f"Training Prophet for {feature}...")
        
        # Prepare data
        prophet_df = df[['timestamp', feature]].rename(
            columns={'timestamp': 'ds', feature: 'y'}
        )
        
        # Remove outliers
        q1 = prophet_df['y'].quantile(0.25)
        q3 = prophet_df['y'].quantile(0.75)
        iqr = q3 - q1
        prophet_df = prophet_df[
            (prophet_df['y'] >= q1 - 1.5 * iqr) & 
            (prophet_df['y'] <= q3 + 1.5 * iqr)
        ]
        
        # Train model
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=True
        )
        model.fit(prophet_df)
        models[feature] = model
        
        # Plot components
        row = idx // 2
        col = idx % 2
        
        # Make future dataframe for visualization
        future = model.make_future_dataframe(periods=168, freq='H')
        forecast = model.predict(future)
        
        axes[row, col].plot(prophet_df['ds'][-168:], prophet_df['y'][-168:], 'b.', markersize=1)
        axes[row, col].plot(forecast['ds'][-168:], forecast['yhat'][-168:], 'r-')
        axes[row, col].fill_between(
            forecast['ds'][-168:],
            forecast['yhat_lower'][-168:],
            forecast['yhat_upper'][-168:],
            color='r', alpha=0.1
        )
        axes[row, col].set_title(f'{feature.capitalize()} - Actual vs Predicted')
        axes[row, col].set_xlabel('Date')
        axes[row, col].set_ylabel(feature)
    
    plt.tight_layout()
    plt.show()
    
    return models

# prophet_models = train_prophet_models(df)

print("Prophet training function ready!")

# Cell 9: Save Prophet Models
def save_prophet_models(models, path='../model/saved'):
    """Save Prophet models"""
    os.makedirs(path, exist_ok=True)
    
    for feature, model in models.items():
        joblib.dump(model, f'{path}/prophet_{feature}.pkl')
    
    print(f"Prophet models saved to {path}")

# save_prophet_models(prophet_models)

print("Save function ready!")

# Cell 10: Model Comparison
def compare_models(df, lstm_predictions, prophet_predictions):
    """Compare LSTM and Prophet predictions"""
    features = ['temperature', 'humidity', 'wind_speed', 'cloud_cover']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    for idx, feature in enumerate(features):
        row = idx // 2
        col = idx % 2
        
        # Plot actual vs predictions
        axes[row, col].plot(df[feature].values[-100:], label='Actual', linewidth=2)
        axes[row, col].plot(lstm_predictions[feature], label='LSTM', linestyle='--')
        axes[row, col].plot(prophet_predictions[feature], label='Prophet', linestyle='-.')
        
        axes[row, col].set_title(f'{feature.capitalize()} - Model Comparison')
        axes[row, col].set_xlabel('Time Steps')
        axes[row, col].set_ylabel(feature)
        axes[row, col].legend()
    
    plt.tight_layout()
    plt.show()

print("Model comparison function ready!")

# Cell 11: Final Training Pipeline
def main_pipeline(data_path='../data/weather_history.csv'):
    """Complete training pipeline"""
    print("=" * 50)
    print("SkyVision Model Training Pipeline")
    print("=" * 50)
    
    # Load data
    print("\n1. Loading data...")
    df = load_weather_data(data_path)
    print(f"   Loaded {len(df)} records")
    
    # Visualize data
    print("\n2. Visualizing data...")
    plot_weather_data(df)
    
    # Train LSTM
    print("\n3. Training LSTM model...")
    X_train, X_test, y_train, y_test, scaler_X, scaler_y = prepare_lstm_data(df)
    lstm_model, history = build_and_train_lstm(X_train, y_train, X_test, y_test)
    
    # Evaluate LSTM
    print("\n4. Evaluating LSTM model...")
    y_pred, y_true = evaluate_lstm(lstm_model, history, X_test, y_test, scaler_y)
    
    # Save LSTM
    print("\n5. Saving LSTM model...")
    save_lstm_model(lstm_model, scaler_X, scaler_y)
    
    # Train Prophet
    print("\n6. Training Prophet models...")
    prophet_models = train_prophet_models(df)
    
    # Save Prophet
    print("\n7. Saving Prophet models...")
    save_prophet_models(prophet_models)
    
    print("\n" + "=" * 50)
    print("Training pipeline completed successfully!")
    print("=" * 50)
    
    return lstm_model, prophet_models, scaler_X, scaler_y

# Uncomment to run the full pipeline:
# lstm_model, prophet_models, scaler_X, scaler_y = main_pipeline()

print("\nTraining notebook ready! Uncomment the pipeline to run.")
print("Make sure to adjust data_path to your dataset location.")